# Intermediate 11 — Adversarial Authorization Testing for Agents

## Scenario: claims-processing agent platform

We will attack a simulated enterprise platform containing:

```text
Users
  ↓ delegate
Claims Agent
  ↓ may delegate
Research Agent
  ↓ invokes
MCP / Tools
  ↓
Claims + Payments APIs
```

The objective is defensive: **discover authorization weaknesses before an attacker does**.

We test decisions *and side effects*. A system that says `DENY` while still executing the operation is broken.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
import hashlib, hmac, json, random, uuid, copy, time
import pandas as pd
import networkx as nx

NOW=datetime.now(timezone.utc)
random.seed(11)


## 1 — Define the security invariants

In [ ]:
INVARIANTS=[
 "cross_tenant_access_is_denied",
 "child_delegation_never_exceeds_parent",
 "expired_delegation_is_denied",
 "unapproved_workload_is_denied",
 "denied_operation_has_no_side_effect",
 "critical_parameters_are_bound_to_authorization",
 "non_redelegable_authority_cannot_be_redelegated",
]
INVARIANTS


## 2 — Build a small platform model

In [ ]:
@dataclass
class Principal:
    id:str
    tenant:str
    permissions:set[str]

@dataclass
class Agent:
    id:str
    tenant:str
    workload_approved:bool
    ambient_permissions:set[str]=field(default_factory=set)

@dataclass
class Resource:
    id:str
    tenant:str
    kind:str

alice=Principal("user:alice","acme",{"claim.read","claim.update"})
mallory=Principal("user:mallory","evil",{"claim.read"})
claims=Agent("agent:claims","acme",True,{"claim.read","claim.update","payment.create"})
research=Agent("agent:research","acme",True,{"document.read"})
claim483=Resource("claim:483","acme","claim")


## 3 — A hardened authorization function

In [ ]:
def authorize(*, principal, agent, action, resource, delegation,
              expected_audience="api://claims", token=None,
              fail_closed=True):
    try:
        if principal.tenant != resource.tenant:
            return False,"TENANT_MISMATCH"
        if agent.tenant != resource.tenant:
            return False,"AGENT_TENANT_MISMATCH"
        if not agent.workload_approved:
            return False,"UNAPPROVED_WORKLOAD"
        if not delegation["active"]:
            return False,"DELEGATION_INACTIVE"
        if delegation["expires_at"] <= NOW:
            return False,"DELEGATION_EXPIRED"
        if delegation["delegatee"] != agent.id:
            return False,"WRONG_DELEGATEE"
        if action not in delegation["actions"]:
            return False,"ACTION_OUT_OF_SCOPE"
        if resource.id not in delegation["resources"]:
            return False,"RESOURCE_OUT_OF_SCOPE"
        if token and token["aud"] != expected_audience:
            return False,"INVALID_AUDIENCE"
        return True,"ALLOW"
    except Exception:
        return (False,"AUTHZ_ERROR") if fail_closed else (True,"FAIL_OPEN")


## 4 — Baseline valid delegation

In [ ]:
delegation={
 "id":"del:1",
 "delegator":"user:alice",
 "delegatee":"agent:claims",
 "actions":{"claim.read","claim.update"},
 "resources":{"claim:483"},
 "active":True,
 "redelegable":False,
 "max_depth":1,
 "expires_at":NOW+timedelta(hours=1)
}
authorize(principal=alice,agent=claims,action="claim.read",
          resource=claim483,delegation=delegation)


## 5 — Attack: trusted-field identity spoofing

In [ ]:
request_body={
 "agent_id":"agent:finance",
 "acting_for":"user:ceo",
 "action":"payment.approve"
}
print("Caller-controlled identity claims:",request_body)
print("Rule: authorization identity must come from authenticated/delegated evidence, not these fields.")


## 6 — Attack: wrong-audience token substitution

In [ ]:
token={"sub":"agent:claims","iss":"https://id.example","aud":"api://analytics"}
authorize(principal=alice,agent=claims,action="claim.read",
          resource=claim483,delegation=delegation,token=token)


## 7 — Attack: expired token

In [ ]:
def validate_exp(exp):
    return exp > NOW.timestamp()

print("expired accepted?",validate_exp((NOW-timedelta(minutes=1)).timestamp()))


## 8 — Attack: replay

In [ ]:
seen=set()
def consume_once(jti):
    if jti in seen:
        return False
    seen.add(jti)
    return True

print(consume_once("token-42"))
print(consume_once("token-42"))


## 9 — Attack: confused deputy

In [ ]:
def vulnerable_deputy(agent, requested_action):
    # BUG: uses agent ambient authority without checking caller delegation.
    return requested_action in agent.ambient_permissions

print("Alice delegated:",delegation["actions"])
print("Agent ambient:",claims.ambient_permissions)
print("Can vulnerable deputy create payment?",vulnerable_deputy(claims,"payment.create"))


## 10 — Fix confused deputy

In [ ]:
allowed,reason=authorize(
 principal=alice,agent=claims,action="payment.create",
 resource=Resource("claim:483","acme","claim"),
 delegation=delegation
)
print(allowed,reason)


## 11 — Attack: delegation scope escalation

In [ ]:
parent={"actions":{"claim.read"},"resources":{"claim:483"}}
child={"actions":{"claim.read","claim.delete"},"resources":{"claim:483"}}
print("escalated:",not child["actions"].issubset(parent["actions"]))


## 12 — Attack: resource expansion

In [ ]:
child2={"actions":{"claim.read"},"resources":{"*"}}
print("resource expansion:", "*" in child2["resources"] and "*" not in parent["resources"])


## 13 — Attack: actor substitution

In [ ]:
stolen=copy.deepcopy(delegation)
attacker_agent=Agent("agent:attacker","acme",True)
print(authorize(principal=alice,agent=attacker_agent,action="claim.read",
                resource=claim483,delegation=stolen))


## 14 — Attack: illegal re-delegation

In [ ]:
def can_redelegate(parent_delegation):
    return parent_delegation.get("redelegable",False)

print("redelegation allowed?",can_redelegate(delegation))


## 15 — Attack: delegation depth

In [ ]:
chain=["user:alice","agent:claims","agent:research"]
depth=len(chain)-1
print("depth",depth,"max",delegation["max_depth"],
      "violation",depth>delegation["max_depth"])


## 16 — Attack: cross-tenant IDOR/BOLA

In [ ]:
evil_claim=Resource("claim:999","evil","claim")
print(authorize(principal=alice,agent=claims,action="claim.read",
                resource=evil_claim,delegation=delegation))


## 17 — Enumerate object IDs

In [ ]:
resources=[
 Resource("claim:481","acme","claim"),
 Resource("claim:482","evil","claim"),
 Resource("claim:483","acme","claim"),
]
for r in resources:
    d=copy.deepcopy(delegation)
    d["resources"]={r.id}
    print(r.id,authorize(principal=alice,agent=claims,action="claim.read",
                         resource=r,delegation=d)[0])


## 18 — Attack: PEP bypass

In [ ]:
paths=pd.DataFrame([
 {"path":"agent->gateway->claims-api","pep":True},
 {"path":"agent->internal-claims-api","pep":False},
 {"path":"agent->mcp->claims-api","pep":True},
 {"path":"queue->worker->claims-api","pep":False},
])
paths[~paths.pep]


## 19 — Attack: fail-open

In [ ]:
def broken_authorize(simulate_failure,fail_closed):
    try:
        if simulate_failure:
            raise TimeoutError("PDP timeout")
        return True
    except Exception:
        return False if fail_closed else True

print("secure:",broken_authorize(True,True))
print("vulnerable:",broken_authorize(True,False))


## 20 — Attack: stale authorization cache

In [ ]:
cache={
 ("agent:claims","claim.read","claim:483"):{"allowed":True,"policy":"v1"}
}
delegation_revoked=True
cached=cache[("agent:claims","claim.read","claim:483")]["allowed"]
print("revoked:",delegation_revoked,"cached allow:",cached)


## 21 — Attack: parameter swap

In [ ]:
def digest(tool,operation,resource,args):
    payload=json.dumps({
      "tool":tool,"operation":operation,"resource":resource,"args":args
    },sort_keys=True,separators=(",",":"))
    return hashlib.sha256(payload.encode()).hexdigest()

approved=digest("payments","create","account:42",{"amount":100,"currency":"CAD"})
executed=digest("payments","create","account:42",{"amount":10000,"currency":"CAD"})
print("bound transaction unchanged?",approved==executed)


## 22 — Attack: tool substitution

In [ ]:
approved_tool={"tool_id":"payments.create","server_id":"mcp:finance","schema_hash":"abc"}
observed_tool={"tool_id":"payments.create","server_id":"mcp:evil","schema_hash":"abc"}
print("substitution:",approved_tool["server_id"]!=observed_tool["server_id"])


## 23 — Attack: MCP definition drift

In [ ]:
observed2={"tool_id":"payments.create","server_id":"mcp:finance","schema_hash":"changed"}
print("definition changed:",approved_tool["schema_hash"]!=observed2["schema_hash"])


## 24 — Attack: agent-to-agent authority laundering

In [ ]:
authority=nx.DiGraph()
authority.add_edges_from([
 ("user:alice","agent:A"),
 ("agent:A","agent:B"),
 ("agent:B","perm:payroll.read"),
])
print("Alice can reach payroll via agents:",
      nx.has_path(authority,"user:alice","perm:payroll.read"))
print("Question: was that transitive authority actually delegated?")


## 25 — Prompt injection is authorization pressure

In [ ]:
model_output={
 "tool":"admin.export_all",
 "resource":"tenant:other",
 "arguments":{"include_secrets":True}
}
print("Treat model output as an untrusted authorization request:")
print(model_output)


## 26 — TOCTOU

In [ ]:
state={"delegation_active":True}
checked=state["delegation_active"]
state["delegation_active"]=False  # revoked before use
executed_based_on_old_check=checked
print("executed using stale check?",executed_based_on_old_check)


## 27 — One-time approval race

In [ ]:
approval={"uses_remaining":1}

# Simulated race: two workers both observe the old value before either commits.
worker1_saw=approval["uses_remaining"]
worker2_saw=approval["uses_remaining"]
print("both think usable:",worker1_saw>0 and worker2_saw>0)
print("Fix with atomic/transactional consume.")


## 28 — Default-allow bug

In [ ]:
def unsafe_policy(action):
    known={"claim.read":True,"claim.delete":False}
    return known.get(action,True)  # BUG

print("unknown action allowed?",unsafe_policy("admin.nuke"))


## 29 — Security oracle: decision AND side effect

In [ ]:
def assert_attack(name,decision,side_effect,expected="deny"):
    passed=(decision==expected and side_effect is False)
    return {"attack":name,"expected":expected,"decision":decision,
            "side_effect":side_effect,"passed":passed}

assert_attack("cross_tenant","deny",False)


## 30 — Metamorphic tests

In [ ]:
base={
 "tenant":"acme","resource_tenant":"acme",
 "workload_approved":True,"delegation_active":True
}

mutations=[
 ("change_tenant",{**base,"resource_tenant":"evil"},"deny"),
 ("remove_attestation",{**base,"workload_approved":False},"deny"),
 ("expire_delegation",{**base,"delegation_active":False},"deny"),
]
mutations


## 31 — Property: tenant isolation

In [ ]:
tenants=["acme","beta","gamma"]
violations=[]
for principal_tenant in tenants:
    for resource_tenant in tenants:
        allowed=principal_tenant==resource_tenant
        if principal_tenant!=resource_tenant and allowed:
            violations.append((principal_tenant,resource_tenant))
print("violations:",violations)


## 32 — Lightweight security fuzzing

In [ ]:
actions=["claim.read","claim.update","claim.delete","unknown.action"]
tenant_values=["acme","evil",None]
cases=[]
for _ in range(100):
    p=random.choice(tenant_values)
    r=random.choice(tenant_values)
    a=random.choice(actions)
    expected_allow=(p=="acme" and r=="acme" and a in {"claim.read","claim.update"})
    cases.append({"principal_tenant":p,"resource_tenant":r,"action":a,
                  "expected_allow":expected_allow})
pd.DataFrame(cases).head()


## 33 — Policy mutation testing

In [ ]:
secure_policy={
 "require_same_tenant":True,
 "require_workload":True,
 "fail_closed":True,
 "allowed_resources":["claim:483"]
}
mutants=[
 {**secure_policy,"require_same_tenant":False},
 {**secure_policy,"require_workload":False},
 {**secure_policy,"fail_closed":False},
 {**secure_policy,"allowed_resources":["*"]},
]
pd.DataFrame(mutants)


A strong regression suite should **kill** each security-relevant mutant. If removing the tenant check causes no test failure, tenant isolation is not actually protected by your tests.

## 34 — Attack catalog

In [ ]:
ATTACKS=[
 ("identity_spoofing","identity"),
 ("token_substitution","token"),
 ("replay","token"),
 ("confused_deputy","delegation"),
 ("scope_escalation","delegation"),
 ("cross_tenant_idor","resource"),
 ("pep_bypass","enforcement"),
 ("fail_open","enforcement"),
 ("stale_cache","runtime"),
 ("parameter_swap","tool"),
 ("mcp_substitution","tool"),
 ("authority_laundering","multi-agent"),
 ("toctou","runtime"),
]
pd.DataFrame(ATTACKS,columns=["attack","category"])


## 35 — Coverage score

In [ ]:
required_categories={
 "identity","token","delegation","resource","policy",
 "enforcement","tool","runtime","multi-agent"
}
covered=set(x[1] for x in ATTACKS)
print("coverage:",len(covered & required_categories),"/",len(required_categories))
print("missing:",required_categories-covered)


## 36 — Severity scoring

In [ ]:
WEIGHT={"low":1,"medium":3,"high":7,"critical":10}
findings=[
 {"id":"PEP-1","severity":"critical","exploitability":1.0,"reachability":1.0},
 {"id":"CACHE-1","severity":"high","exploitability":0.8,"reachability":0.8},
]
for f in findings:
    f["score"]=WEIGHT[f["severity"]]*f["exploitability"]*f["reachability"]
pd.DataFrame(findings).sort_values("score",ascending=False)


## 37 — Regression report

In [ ]:
report={
 "run_id":str(uuid.uuid4()),
 "timestamp":NOW.isoformat(),
 "policy_version":"v20",
 "tests":35,
 "failures":findings,
 "critical_invariants":{
   "tenant_isolation":"PASS",
   "delegation_attenuation":"PASS",
   "pep_coverage":"FAIL",
   "revocation_freshness":"FAIL"
 }
}
print(json.dumps(report,indent=2))


## 38 — OPA lab

Files:

```text
policies/opa/adversarial.rego
policies/opa/adversarial_test.rego
```

Run:

```bash
opa test policies/opa -v --fail-on-empty
```

Then deliberately mutate:

```text
remove tenant check
remove workload check
remove delegatee binding
```

The tests should fail. Add a test for every mutation that survives.


## 39 — Cedar lab

Use:

```text
policies/cedar/adversarial.cedar
policies/cedar/adversarial_fixed.cedar
```

Test the semantics:

```text
no permit -> Deny
permit -> Allow
permit + matching forbid -> Deny
wrong tenant -> Deny
unapproved workload -> Deny
expired/stale context -> Deny
```

Also validate policies against a Cedar schema in your production implementation. Cedar's schema validation is an important defense against malformed policy/entity assumptions.


## 40 — OpenFGA lab

Use:

```text
policies/openfga/model.fga
policies/openfga/adversarial.fga.yaml
```

Run OpenFGA model tests with the FGA CLI.

Extend the suite with:

```text
cross-tenant relationships
removed tuples
unexpected parent inheritance
sub-agent without delegation
ListObjects negative tests
ListUsers negative tests
contextual conditions
```

OpenFGA's current testing format supports `Check`, `ListObjects`, and `ListUsers` assertions and is designed for CI/CD model testing.


## 41 — CI/CD security gate

A production pipeline can require:

```text
policy syntax/schema validation
        ↓
OPA/Cedar/OpenFGA unit tests
        ↓
negative attack fixtures
        ↓
property-based tests
        ↓
security mutations
        ↓
integration tests with real PEP
        ↓
canary
        ↓
authorization telemetry
```

Example release rule:

```text
0 critical invariant failures
0 cross-tenant bypasses
0 PEP bypasses
100% required negative suites executed
mutation score >= organizational threshold
```

Thresholds should be risk-based rather than copied blindly.


## 42 — Capstone exercise

Build an adversarial authorization test plan for a production agent that can:

```text
read customer records
update claims
search enterprise knowledge
invoke an MCP server
request payments with human approval
delegate research to a sub-agent
```

Your plan must include:

1. security invariants;
2. trust boundaries;
3. at least 25 negative scenarios;
4. token attacks;
5. delegation attacks;
6. tenant/resource attacks;
7. tool/MCP attacks;
8. PEP bypass tests;
9. TOCTOU/race tests;
10. property-based tests;
11. mutations;
12. CI gates;
13. telemetry requirements;
14. remediation ownership;
15. permanent regression tests.

The deliverable should demonstrate that the architecture remains secure when components are malicious, compromised, stale, misconfigured, or simply wrong.


# Review questions

1. Why is policy correctness different from authorization-system security?
2. What is a security invariant?
3. What is a confused deputy?
4. Why is ambient agent authority dangerous?
5. What is token substitution?
6. Why must audience be validated?
7. What authorization artifacts are replayable?
8. What is a downgrade path?
9. Why must caller-controlled identity fields be ignored?
10. How should logical agent and workload identity interact?
11. What is delegation attenuation?
12. Why validate the complete delegation chain?
13. What is actor substitution?
14. How can resource scope expand?
15. Why are cross-tenant tests especially important?
16. What is IDOR/BOLA?
17. What is a PEP bypass?
18. Why can fail-open behavior become a security vulnerability?
19. How can caching break revocation?
20. What is parameter tampering after authorization?
21. Why verify MCP/tool identity?
22. What is authority laundering?
23. Why should model output be treated as untrusted?
24. What is TOCTOU?
25. Why do one-time approvals need atomic enforcement?
26. What is mutation testing?
27. What is a surviving mutant?
28. What is a metamorphic authorization test?
29. Why validate side effects as well as allow/deny?
30. How should authorization security tests enter CI/CD?

# Next course

## Intermediate 12 — Integrating Authorization with LLMs, Agents & Guardrails
